# AI-driven Quiz/Flashcard Generator for Any Topic

This notebook allows you to enter any topic, and it will generate a set of quiz questions or flashcards for study using an LLM (e.g., OpenAI or compatible model).

---

**Instructions:**
1. Enter your OpenAI API key (or configure your environment for Ollama/local LLM if preferred).
2. Enter a topic in the input cell.
3. Run the notebook to generate quiz questions or flashcards.


In [8]:
# Import required libraries
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables (if using .env for API key)
load_dotenv(override=True)

# Set your OpenAI API key here or in your .env file
api_key = os.getenv('OPENAI_API_KEY')
ollama_url = "http://localhost:11434/v1"

print (f"API Key: {api_key}")

if not api_key:
    client = OpenAI(api_key="ollama", base_url=ollama_url)
    model="llama3.2:latest"
else:
    model="gpt-3.5-turbo"
    client = OpenAI(api_key=api_key)

# Helper function to generate quiz questions or flashcards
def generate_quiz(topic, num_questions=5, mode="quiz"):
    """
    Generate quiz questions or flashcards for a given topic using OpenAI.
    mode: 'quiz' for Q&A, 'flashcard' for term/definition pairs
    """
    if mode == "quiz":
        prompt = f"""
Generate {num_questions} quiz questions and answers about the topic: '{topic}'.
Format as:
Q1: ...\nA1: ...\nQ2: ...\nA2: ...
"""
    else:
        prompt = f"""
Generate {num_questions} flashcards about the topic: '{topic}'.
Format as:
Front 1: ...\nBack 1: ...\nFront 2: ...\nBack 2: ...
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.7
    )
    return response.choices[0].message.content.strip()


API Key: None


In [5]:
# User input for topic and mode
import ipywidgets as widgets
from IPython.display import display, Markdown

topic_widget = widgets.Text(
    value='',
    placeholder='e.g. The Solar System',
    description='Topic:',
    disabled=False
)

num_questions_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Questions:',
    disabled=False
)

mode_widget = widgets.ToggleButtons(
    options=[('Quiz', 'quiz'), ('Flashcard', 'flashcard')],
    value='quiz',
    description='Mode:',
    disabled=False,
    button_style=''
)

display(topic_widget, num_questions_widget, mode_widget)


Text(value='', description='Topic:', placeholder='e.g. The Solar System')

IntSlider(value=5, description='Questions:', max=20, min=1)

ToggleButtons(description='Mode:', options=(('Quiz', 'quiz'), ('Flashcard', 'flashcard')), value='quiz')

In [ ]:
# Button to trigger generation and display results
generate_button = widgets.Button(description="Generate Quiz/Flashcards", button_style='success')
output = widgets.Output()

def on_generate_clicked(b):
    output.clear_output()
    topic = topic_widget.value.strip()
    num_questions = num_questions_widget.value
    mode = mode_widget.value
    if not topic:
        with output:
            display(Markdown("**Please enter a topic.**"))
        return
    with output:
        display(Markdown(f"### Generating {num_questions} {'quiz questions' if mode=='quiz' else 'flashcards'} for: **{topic}** ..."))
        try:
            result = generate_quiz(topic, num_questions, mode)
            display(Markdown(f"```\n{result}\n```"))
        except Exception as e:
            display(Markdown(f"**Error:** {e}"))

generate_button.on_click(on_generate_clicked)
display(generate_button, output)


Button(button_style='success', description='Generate Quiz/Flashcards', style=ButtonStyle())

Output()